# Lab 21 — RUN CHO BÀI NỘP (T4, cấu hình mặc định)

**Trước khi bắt đầu:** Runtime → Change runtime type → **T4 GPU**.

> Nếu bạn vừa `git push` lên repo, hãy **mở LẠI tab này** (reload URL), đừng chỉ
> reconnect — Colab chỉ đọc mã notebook từ GitHub đúng một lần (F-19).

| Ô | Làm gì | Thời gian (T4, đo thật) |
|---|---|---|
| 1 | clone repo + cài dependency | ~1–2 ph |
| 2 | smoke: import + 115 unit test | ~30 s |
| 3 | khoá cấu hình chạy (mặc định, KHÔNG rút gọn) | tức thì |
| 4 | NB1 + NB2 — mask proof + **đóng băng 2 baseline** | ~20 ph |
| 5 | NB3 — train cấu hình đúng | ~15–25 ph |
| 6 | NB4 — 3 run đối chứng | ~45–60 ph |
| 7 | NB5 — eval 4 nhóm + phán quyết | ~21 ph |
| 8 | NB6 — merge + hot-swap *(thưởng B1, +3đ)* | ~10 ph |
| 9 | gatekeeper `verify.py` | ~10 s |
| 10 | đóng gói + tải `results/` + adapter về máy | ~30 s |

Chạy tuần tự. Ô 4→8 mỗi ô là một chặng độc lập: nếu đứt kết nối, chỉ chạy lại
**ô đang dở** — adapter nào đã lưu sẽ được bỏ qua, không train lại từ đầu.

Giữ tab hoạt động trong suốt quá trình (đừng để máy sleep).

---

### Làm sao biết một ô đã chạy thành công

**Đừng tin dấu ✓.** Colab đánh ✓ khi ô kết thúc mà Python không ném lỗi, còn cú pháp
`!python ...` thì không ném lỗi khi chương trình con chết. Một chặng train hỏng vẫn
hiện ✓ kèm thời gian, y hệt một chặng thành công.

Nên các ô ở đây tự khai báo mã thoát của mình. Dòng **cuối cùng** của output là thứ
cần đọc:

| Thấy gì | Nghĩa là |
|---|---|
| `✅ NB3 xong` | thành công — đi tiếp |
| ô **đỏ**, `RuntimeError: NB3 HỎNG (exit 1)` | hỏng — nguyên nhân nằm ở traceback ngay phía trên |

Cảnh báo riêng cho ô 4→8: `colab_run.py` in bảng `stage timings` **cả khi hỏng**. Bảng
thời gian không phải bằng chứng thành công; dòng `!! NBx FAILED (exit N)` ngay trên nó
mới là. Ô 9 là ngoại lệ duy nhất — nó cố tình không ném lỗi, xem chú thích trong ô.

Lỗi hay gặp nhất ở ô 6: `CUDA out of memory`. Chạy lại chính ô đó; nếu vẫn lặp lại thì
Runtime → Restart session rồi chạy lại từ ô 3 (adapter đã lưu vẫn được giữ).

In [ ]:
# @title 1. Setup — clone repo + cài dependency
import os, subprocess, sys

REPO = "https://github.com/Thanhhuy2000/Day21-Track3-Finetuning-2A202601802-NguyenThanhHuy.git"
DIR  = "Day21-Track3-Finetuning-2A202601802-NguyenThanhHuy"

if not os.path.exists("/content/" + DIR):
    subprocess.run(["git", "clone", "-q", REPO, "/content/" + DIR], check=True)
os.chdir("/content/" + DIR)
subprocess.run(["git", "pull", "-q"], check=False)
sys.path.insert(0, "src")

# Cài từ requirements.txt — MỘT nguồn sự thật duy nhất (F-20). Danh sách chép tay là
# cách pin torchao>=0.16 bị lệch giữa các file, và lỗi đó không nổ ở đây mà nổ 10 phút
# sau, bên trong get_peft_model(). torch có sẵn trên Colab nên dòng đó là no-op.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
               check=True)

import torch
print("commit :", subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                                 capture_output=True, text=True).stdout.strip())
print("GPU    :", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "NONE — Runtime > Change runtime type > T4 GPU")
if torch.cuda.is_available():
    print("VRAM   : %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1024**3))
print("torch  :", torch.__version__)

In [ ]:
# @title 2. Smoke — import + dữ liệu + unit test (chưa cần GPU)
STAGE = "Smoke"
!python scripts/verify.py --smoke

if _exit_code:
    raise RuntimeError(f"{STAGE} HỎNG (exit {_exit_code}) — đọc traceback phía trên. "
                       f"Đừng chạy ô tiếp theo cho tới khi ô này xanh.")
print(f"✅ {STAGE} xong")

In [ ]:
# @title 3. Khoá cấu hình cho bài nộp — mặc định, KHÔNG rút gọn
import os

os.environ["COMPUTE_TIER"] = "T4"            # unsloth/Qwen3.5-4B
os.environ["MASK_MODE"]    = "assistant-only"
os.environ["EPOCHS"]       = "2"             # 30 optimizer step
os.environ.pop("EVAL_LIMIT", None)           # <- eval ĐẦY ĐỦ 50 mẫu. Bài nộp phải vậy.

from labkit import device
from labkit.config import get_tier

print(device.banner())
t = get_tier()
print(f"tier={t.name}  model={t.model_id}  max_length={t.max_length}  "
      f"effective_batch={t.effective_batch}")
print("EVAL_LIMIT =", os.environ.get("EVAL_LIMIT") or "full (50)")

# Ghi lại môi trường thật — grader đối chiếu, và report cần số máy thật.
import pathlib, subprocess, sys
pathlib.Path("results").mkdir(exist_ok=True)
with open("results/ENVIRONMENT.txt", "w", encoding="utf-8") as f:
    f.write(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
    f.write("\n\n=== pip freeze ===\n")
    f.write(subprocess.run([sys.executable, "-m", "pip", "freeze"],
                           capture_output=True, text=True).stdout)
    f.write("\n=== commit ===\n")
    f.write(subprocess.run(["git", "rev-parse", "HEAD"],
                           capture_output=True, text=True).stdout)
print("-> results/ENVIRONMENT.txt")

In [ ]:
# @title 4. NB1 + NB2 — mask proof, rồi ĐÓNG BĂNG hai baseline trước khi train (~20 ph)
STAGE = "NB1+NB2"
# Thứ tự này là điểm mấu chốt của lab: baseline phải đo TRƯỚC khi có adapter,
# nếu không thì phép so sánh không còn ý nghĩa gì.
!python scripts/colab_run.py nb1 nb2

if _exit_code:
    raise RuntimeError(f"{STAGE} HỎNG (exit {_exit_code}) — đọc traceback phía trên. "
                       f"Đừng chạy ô tiếp theo cho tới khi ô này xanh.")
print(f"✅ {STAGE} xong")

In [ ]:
# @title 5. NB3 — train cấu hình đúng (text-linear, r=16, LR 1e-4) (~15–25 ph)
STAGE = "NB3"
!python scripts/colab_run.py nb3

if _exit_code:
    raise RuntimeError(f"{STAGE} HỎNG (exit {_exit_code}) — đọc traceback phía trên. "
                       f"Đừng chạy ô tiếp theo cho tới khi ô này xanh.")
print(f"✅ {STAGE} xong")

In [ ]:
# @title 6. NB4 — ba run đối chứng: attn_only · wrong_lr · qlora (~45–60 ph)
STAGE = "NB4"
# Cùng số step, cùng ngân sách tham số, mỗi run đổi ĐÚNG một biến.
# Nếu ô này đứt giữa chừng: chạy lại chính nó — adapter đã lưu sẽ được bỏ qua.
# Muốn train lại tất cả: FORCE_RETRAIN=1. Train lại đúng một run: ONLY=qlora.
!python scripts/colab_run.py nb4

if _exit_code:
    raise RuntimeError(f"{STAGE} HỎNG (exit {_exit_code}) — đọc traceback phía trên. "
                       f"Đừng chạy ô tiếp theo cho tới khi ô này xanh.")
print(f"✅ {STAGE} xong")

In [ ]:
# @title 7. NB5 — eval 4 nhóm + cổng hồi quy + phán quyết (~21 ph)
STAGE = "NB5"
!python scripts/colab_run.py nb5

if _exit_code:
    raise RuntimeError(f"{STAGE} HỎNG (exit {_exit_code}) — đọc traceback phía trên. "
                       f"Đừng chạy ô tiếp theo cho tới khi ô này xanh.")
print(f"✅ {STAGE} xong")

In [ ]:
# @title 8. NB6 — merge + hot-swap adapter (THƯỞNG B1, +3 điểm) (~10 ph)
STAGE = "NB6"
!python scripts/colab_run.py nb6

if _exit_code:
    raise RuntimeError(f"{STAGE} HỎNG (exit {_exit_code}) — đọc traceback phía trên. "
                       f"Đừng chạy ô tiếp theo cho tới khi ô này xanh.")
print(f"✅ {STAGE} xong")

In [ ]:
# @title 9. Gatekeeper — kiểm artefact VÀ tính liêm chính của phép so sánh
# Ô này CỐ TÌNH không ném lỗi. Ở thời điểm này verify.py chắc chắn báo FAIL ít nhất
# một mục: "REPORT.md filled in" — report chưa viết. Đó là đúng, không phải hỏng.
# Cái cần đọc là các mục khác: mask proof, khớp ngân sách tham số, checksum eval,
# SHA của prompt (b), và (b) > (a).
!python scripts/verify.py

print("\n" + "=" * 30 + " results/ " + "=" * 30)
!ls -la results/
for name in ["mask_proof.json", "template_check.json", "token_stats.json",
             "baselines_frozen.json", "verdict.json", "autopsy.json",
             "qualitative.json", "merge_check.json"]:
    print("\n---- " + name + " ----")
    !cat results/{name} 2>/dev/null || echo "(thieu)"
print("\n---- runs.csv ----")
!cat results/runs.csv 2>/dev/null

In [ ]:
# @title 10. Đóng gói + tải về máy
import shutil, os, pathlib

os.makedirs("/content/pack/results", exist_ok=True)
os.makedirs("/content/pack/adapters", exist_ok=True)

for p in pathlib.Path("results").glob("*"):
    if p.is_file():
        shutil.copy2(p, "/content/pack/results/" + p.name)

# Chỉ adapter chính (Option A của rubric).
src = pathlib.Path("adapters/correct")
if src.exists():
    shutil.copytree(src, "/content/pack/adapters/correct", dirs_exist_ok=True)

shutil.make_archive("/content/lab21_results", "zip", "/content/pack")
size = os.path.getsize("/content/lab21_results.zip") / 1024**2
print(f"lab21_results.zip - {size:.1f} MB")
!unzip -l /content/lab21_results.zip

from google.colab import files
files.download("/content/lab21_results.zip")